# Setup and reload

In [1]:
import sys, os
from pathlib import Path
import logging

def find_project_root(marker="backend", start=None):
    current = Path(start or os.getcwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / marker).is_dir():
            return candidate
    raise RuntimeError(f"Could not find a '{marker}' folder above {current}")

PROJECT_ROOT = find_project_root()
print(f"Project root: {PROJECT_ROOT}")

sys.path.insert(0, str(PROJECT_ROOT / "backend"))
sys.path.insert(0, str(PROJECT_ROOT / "backend" / "src"))
sys.path.insert(0, str(PROJECT_ROOT / "backend" / "scripts"))

logging.basicConfig(level=logging.INFO, format="%(message)s")
logging.getLogger("httpx").setLevel(logging.WARNING)

from qdrant_client import QdrantClient
from qdrant_client.http import models as qmodels
from fastembed import SparseTextEmbedding
import openai

from medrag.embeddings.qdrant_client import TEXT_COLLECTION, DENSE_VECTOR_NAME, SPARSE_VECTOR_NAME

client = QdrantClient(url="http://localhost:6333")
client.get_collections()

sparse_model = SparseTextEmbedding(model_name="Qdrant/bm25")

print("Setup complete")

Project root: C:\Users\DELL\Desktop\medrag


c:\Users\DELL\Desktop\medrag\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Setup complete


# Encode a test query both ways, run each retrieval independently

In [2]:
import os

openai_client = openai.OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

def embed_query_dense(text: str):
    response = openai_client.embeddings.create(model="text-embedding-3-small", input=text)
    return response.data[0].embedding

def embed_query_sparse(text: str):
    return list(sparse_model.embed([text]))[0]

query_text = "hypertension treatment first line medication"

dense_query_vec = embed_query_dense(query_text)
sparse_query_vec = embed_query_sparse(query_text)

dense_results = client.query_points(
    collection_name=TEXT_COLLECTION,
    query=dense_query_vec,
    using=DENSE_VECTOR_NAME,
    limit=10,
).points

sparse_results = client.query_points(
    collection_name=TEXT_COLLECTION,
    query=qmodels.SparseVector(
        indices=sparse_query_vec.indices.tolist(),
        values=sparse_query_vec.values.tolist(),
    ),
    using=SPARSE_VECTOR_NAME,
    limit=10,
).points

print("DENSE results:")
for rank, r in enumerate(dense_results, 1):
    print(f"  {rank}. score={r.score:.4f}  chunk_id={r.payload['chunk_id']}")

print("\nSPARSE results:")
for rank, r in enumerate(sparse_results, 1):
    print(f"  {rank}. score={r.score:.4f}  chunk_id={r.payload['chunk_id']}")

DENSE results:
  1. score=0.6226  chunk_id=hypertension_who_table_125
  2. score=0.6124  chunk_id=hypertension_who_text_91
  3. score=0.6012  chunk_id=hypertension_who_text_72
  4. score=0.5888  chunk_id=hypertension_who_text_76
  5. score=0.5866  chunk_id=hypertension_who_text_31
  6. score=0.5780  chunk_id=hypertension_who_text_90
  7. score=0.5773  chunk_id=hypertension_who_table_123
  8. score=0.5730  chunk_id=hypertension_who_text_74
  9. score=0.5684  chunk_id=hypertension_who_text_71
  10. score=0.5666  chunk_id=hypertension_who_table_126

SPARSE results:
  1. score=32.6898  chunk_id=hypertension_who_text_111
  2. score=29.6721  chunk_id=hypertension_who_text_12
  3. score=29.3965  chunk_id=hypertension_who_table_125
  4. score=28.8271  chunk_id=hypertension_who_text_91
  5. score=28.6311  chunk_id=hypertension_who_text_112
  6. score=28.4860  chunk_id=hypertension_who_table_115
  7. score=28.4197  chunk_id=hypertension_who_text_76
  8. score=26.2282  chunk_id=hypertension_who_t

# Hand-implement RRF, fuse the two lists

In [3]:
from collections import defaultdict

def reciprocal_rank_fusion(result_lists, k=60):
    """result_lists: list of ranked result lists (each a list of objects
    with .payload['chunk_id']). Returns chunk_id -> fused_score dict,
    sorted descending by fused score."""
    fused_scores = defaultdict(float)
    chunk_payloads = {}

    for results in result_lists:
        for rank, r in enumerate(results, start=1):
            chunk_id = r.payload["chunk_id"]
            fused_scores[chunk_id] += 1.0 / (k + rank)
            chunk_payloads[chunk_id] = r.payload

    ranked = sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    return ranked, chunk_payloads

fused_ranked, chunk_payloads = reciprocal_rank_fusion([dense_results, sparse_results], k=60)

print("FUSED (RRF) results:")
for rank, (chunk_id, score) in enumerate(fused_ranked[:10], 1):
    print(f"  {rank}. fused_score={score:.5f}  chunk_id={chunk_id}")

FUSED (RRF) results:
  1. fused_score=0.03227  chunk_id=hypertension_who_table_125
  2. fused_score=0.03175  chunk_id=hypertension_who_text_91
  3. fused_score=0.03055  chunk_id=hypertension_who_text_76
  4. fused_score=0.02967  chunk_id=hypertension_who_text_31
  5. fused_score=0.01639  chunk_id=hypertension_who_text_111
  6. fused_score=0.01613  chunk_id=hypertension_who_text_12
  7. fused_score=0.01587  chunk_id=hypertension_who_text_72
  8. fused_score=0.01538  chunk_id=hypertension_who_text_112
  9. fused_score=0.01515  chunk_id=hypertension_who_text_90
  10. fused_score=0.01515  chunk_id=hypertension_who_table_115


# Qdrant's native RRF fusion (single request)

In [4]:
native_fused = client.query_points(
    collection_name=TEXT_COLLECTION,
    prefetch=[
        qmodels.Prefetch(query=dense_query_vec, using=DENSE_VECTOR_NAME, limit=10),
        qmodels.Prefetch(
            query=qmodels.SparseVector(
                indices=sparse_query_vec.indices.tolist(),
                values=sparse_query_vec.values.tolist(),
            ),
            using=SPARSE_VECTOR_NAME,
            limit=10,
        ),
    ],
    query=qmodels.FusionQuery(fusion=qmodels.Fusion.RRF),
    limit=10,
).points

print("NATIVE QDRANT RRF results:")
for rank, r in enumerate(native_fused, 1):
    print(f"  {rank}. score={r.score:.5f}  chunk_id={r.payload['chunk_id']}")

NATIVE QDRANT RRF results:
  1. score=0.75000  chunk_id=hypertension_who_table_125
  2. score=0.53333  chunk_id=hypertension_who_text_91
  3. score=0.50000  chunk_id=hypertension_who_text_111
  4. score=0.33333  chunk_id=hypertension_who_text_12
  5. score=0.32500  chunk_id=hypertension_who_text_76
  6. score=0.25758  chunk_id=hypertension_who_text_31
  7. score=0.25000  chunk_id=hypertension_who_text_72
  8. score=0.16667  chunk_id=hypertension_who_text_112
  9. score=0.14286  chunk_id=hypertension_who_table_115
  10. score=0.14286  chunk_id=hypertension_who_text_90


In [5]:
help(qmodels.FusionQuery)

Help on class FusionQuery in module qdrant_client.http.models.models:

class FusionQuery(pydantic.main.BaseModel)
 |  FusionQuery(*, fusion: qdrant_client.http.models.models.Fusion) -> None
 |  
 |  Method resolution order:
 |      FusionQuery
 |      pydantic.main.BaseModel
 |      builtins.object
 |  
 |  Data descriptors defined here:
 |  
 |  __weakref__
 |      list of weak references to the object
 |  
 |  ----------------------------------------------------------------------
 |  Data and other attributes defined here:
 |  
 |  __abstractmethods__ = frozenset()
 |  
 |  __annotations__ = {'fusion': 'Fusion'}
 |  
 |  __class_vars__ = set()
 |  
 |  __private_attributes__ = {}
 |  
 |  __pydantic_complete__ = True
 |  
 |  __pydantic_computed_fields__ = {}
 |  
 |  __pydantic_core_schema__ = {'cls': <class 'qdrant_client.http.models.m...
 |  
 |  __pydantic_custom_init__ = False
 |  
 |  __pydantic_decorators__ = DecoratorInfos(validators={}, field_validato...
 |  
 |  __pydantic_

# Wrap into a hybrid_search() function, test on varied queries

In [6]:
def hybrid_search(query_text: str, limit: int = 10, k: int = 60):
    dense_vec = embed_query_dense(query_text)
    sparse_vec = embed_query_sparse(query_text)

    dense_results = client.query_points(
        collection_name=TEXT_COLLECTION,
        query=dense_vec,
        using=DENSE_VECTOR_NAME,
        limit=limit,
    ).points

    sparse_results = client.query_points(
        collection_name=TEXT_COLLECTION,
        query=qmodels.SparseVector(
            indices=sparse_vec.indices.tolist(),
            values=sparse_vec.values.tolist(),
        ),
        using=SPARSE_VECTOR_NAME,
        limit=limit,
    ).points

    ranked, payloads = reciprocal_rank_fusion([dense_results, sparse_results], k=k)
    return [
        {"chunk_id": chunk_id, "fused_score": score, "payload": payloads[chunk_id]}
        for chunk_id, score in ranked[:limit]
    ]


# Test 1: a drug-name-heavy query (sparse should shine here)
print("=== Query: 'metformin dosage for diabetes' ===")
for r in hybrid_search("metformin dosage for diabetes", limit=5):
    print(f"  {r['fused_score']:.5f}  source={r['payload']['source']}  chunk_id={r['chunk_id']}")

# Test 2: a broader conceptual query (dense should shine here)
print("\n=== Query: 'how does the body regulate blood sugar' ===")
for r in hybrid_search("how does the body regulate blood sugar", limit=5):
    print(f"  {r['fused_score']:.5f}  source={r['payload']['source']}  chunk_id={r['chunk_id']}")

=== Query: 'metformin dosage for diabetes' ===
  0.03126  source=openfda  chunk_id=Metformin Hydrochloride_openfda_0
  0.01639  source=openfda  chunk_id=Metformin Hydrochloride_openfda_1
  0.01639  source=openfda  chunk_id=Synjardy_openfda_26
  0.01613  source=openfda  chunk_id=Metformin Hydrochloride_openfda_2
  0.01613  source=openfda  chunk_id=Jardiance_openfda_23

=== Query: 'how does the body regulate blood sugar' ===
  0.01639  source=who  chunk_id=coronary artery disease+heart failure+hyperlipidemia+stroke_who_text_114
  0.01639  source=who  chunk_id=asthma+copd_who_text_21
  0.01613  source=openfda  chunk_id=Glipizide_openfda_11
  0.01613  source=openfda  chunk_id=Methylprednisolone Sodium Succinate_openfda_7
  0.01587  source=pubmed  chunk_id=42467085_pubmed_0


# Diagnose: dense-only and sparse-only results for the weak query

In [7]:
query_text = "how does the body regulate blood sugar"
dense_vec = embed_query_dense(query_text)
sparse_vec = embed_query_sparse(query_text)

dense_only = client.query_points(
    collection_name=TEXT_COLLECTION, query=dense_vec, using=DENSE_VECTOR_NAME, limit=5,
).points

sparse_only = client.query_points(
    collection_name=TEXT_COLLECTION,
    query=qmodels.SparseVector(indices=sparse_vec.indices.tolist(), values=sparse_vec.values.tolist()),
    using=SPARSE_VECTOR_NAME, limit=5,
).points

print("DENSE ONLY:")
for r in dense_only:
    print(f"  score={r.score:.4f}  source={r.payload['source']}  chunk_id={r.payload['chunk_id']}")
    print(f"    text: {r.payload['raw_text'][:150]}")

print("\nSPARSE ONLY:")
for r in sparse_only:
    print(f"  score={r.score:.4f}  source={r.payload['source']}  chunk_id={r.payload['chunk_id']}")
    print(f"    text: {r.payload['raw_text'][:150]}")

DENSE ONLY:
  score=0.4511  source=who  chunk_id=coronary artery disease+heart failure+hyperlipidemia+stroke_who_text_114
    text: In a meta-analysis of non-diabetic
subjects, those with the highest blood glucose levels had a relative risk for cardiovascular disease
events of 1.26
  score=0.4343  source=openfda  chunk_id=Glipizide_openfda_11
    text: Mechanism of Action The primary mode of action of glipizide in experimental animals appears to be the stimulation of insulin secretion from the beta c
  score=0.4228  source=pubmed  chunk_id=42467085_pubmed_0
    text: Circulating proteins act as important hormonal signals of nutrient intake. We aimed to systematically characterise the time-resolved proteomic respons
  score=0.4219  source=pubmed  chunk_id=42479963_pubmed_0
    text: Emotion regulation and glycaemic responses are closely interconnected, with difficulties in one domain often exacerbating the other. This relationship
  score=0.4168  source=who  chunk_id=diabetes_who_table_